In [3]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import os

In [ ]:
# Download data bulan Juni
import requests

url = "https://github.com/DataTalksClub/nyc-tlc-data/releases/download/yellow/yellow_tripdata_2021-06.csv.gz"
file_path = "../data/raw/yellow_tripdata_2021-06.csv.gz"

# Download file
if not os.path.exists(file_path):
    print("Sedang mengunduh data bulan Juni...")
    response = requests.get(url)
    with open(file_path, 'wb') as f:
        f.write(response.content)
    print("Download Selesai")
else:
    Print("File sudah ada / tidak ditemukan")

In [6]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import os

os.environ['HADOOP_HOME'] = "C:/hadoop"

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("union_june_july") \
    .getOrCreate()

# Baca data bulan juli parquet
df_july = spark.read.parquet('../data/pq/yellow_tripdata_2021-07.parquet')

df_june_raw = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv('../data/raw/yellow_tripdata_2021-06.csv.gz')

print(f"Kolom Juni: {df_june_raw.columns}")
print(f"Kolom Juli: {df_july.columns}")

Kolom Juni: ['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge']
Kolom Juli: ['index', 'VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge']


In [ ]:
# 1. Identifikasi kolom extra di bulan Juli

cols_july = set(df_july.columns)
cols_june = set(df_june_raw.columns)

extra_cols = cols_july - cols_june
print(f"Kolom extra di bulan Juli: {extra_cols}")

In [ ]:
# 2. Buang kolom ekstra dari Juli
df_july_cleaned = df_july.drop(*extra_cols)

# 3. Sinkronisasi Tipe Data (Krusial!)
for col_name, dtype in df_july_cleaned.dtypes:
    df_june_raw = df_june_raw.withColumn(col_name, df_june_raw[col_name].cast(dtype))

# 4. Menggabungkan dengan unionByName
df_combined = df_july_cleaned.unionByName(df_june_raw)

print(f"Skema Akhir: {len(df_combined.columns)} kolom")
print(f"Total Baris Gabungan: {df_combined.count()}")

In [ ]:
# Melihat jumlah data per bulan untuk memastikan keduanya masuk

df_combined.withColumn("month", F.month("tpep_pickup_datetime")) \
            .groupBy("month") \
            .count() \
            .show()

In [ ]:
# Assign lokasi penyimpanan
output_path = '../data/pq/combined'

# Re-partitioning data menjadi 4 file terpisah

df_combined_repartitioned = df_combined.repartition(4)

# Proses Penulisan mode = "overwrite". Artinya jika folder sudah ada, isinya akan diganti
df_combined_repartitioned.write.parquet(output_path, mode="overwrite")

print(f"Data berhasil disimpan di: {output_path}")

In [7]:
# Mencoba membaca kembali hasil simpanan tadi
df_final_check = spark.read.parquet('../data/pq/combined')

print(f"Verifikasi jumlah baris: {df_final_check.count()}")
df_final_check.select("tpep_pickup_datetime", "trip_distance").show(5)

Verifikasi jumlah baris: 5655779
+--------------------+-------------+
|tpep_pickup_datetime|trip_distance|
+--------------------+-------------+
| 2021-07-02 17:02:32|         2.32|
| 2021-07-02 07:59:13|          1.5|
| 2021-07-01 11:42:18|          1.5|
| 2021-07-01 19:08:26|          3.7|
| 2021-07-01 20:33:13|         3.46|
+--------------------+-------------+
only showing top 5 rows


In [8]:
# Temporary View SQL

# Membaca data gabungan yang sudah disimpan
df_combined = spark.read.parquet('../data/pq/combined')

# Mendaftarkan DataFrame sebagai Temporary View dengan memanggil kolom trips_data
df_combined.createOrReplaceTempView('trips_data')

# Menjalankan Query SQL
# 5 Perjalanan dengan ongkos (fare_amount) tertinggi

top_fares = spark.sql("""
    SELECT
        VendorID,
        tpep_pickup_datetime,
        trip_distance,
        fare_amount
    FROM
        trips_data
    ORDER BY
        fare_amount DESC
    LIMIT 10
""")

top_fares.show()

+--------+--------------------+-------------+-----------+
|VendorID|tpep_pickup_datetime|trip_distance|fare_amount|
+--------+--------------------+-------------+-----------+
|       2| 2021-07-27 13:08:58|        25.77|     1320.0|
|       1| 2021-07-17 10:25:22|        323.0|      823.0|
|       1| 2021-06-15 23:57:37|        282.1|      716.0|
|       2| 2021-07-15 19:06:27|          0.0|      700.0|
|       2| 2021-07-15 19:04:29|          0.0|      700.0|
|       1| 2021-06-30 14:31:15|          0.0|     684.07|
|       1| 2021-07-29 15:04:32|          0.0|     655.35|
|       1| 2021-06-24 16:02:40|          0.0|      650.0|
|       1| 2021-07-13 18:23:28|          0.0|      650.0|
|       1| 2021-06-01 14:19:19|          0.0|      650.0|
+--------+--------------------+-------------+-----------+



In [ ]:
# Jika ini lancar dalam < 10 detik, berarti mesin sehat
spark.sql("SELECT COUNT(*) FROM trips_data").show()

+--------+
|count(1)|
+--------+
| 5655779|
+--------+



In [11]:
monthly_trend = spark.sql("""
    SELECT
        date_trunc('month', tpep_pickup_datetime) AS month_revenue,
        COUNT(1) AS total_trips,
        SUM(total_amount) AS revenue
    FROM
        trips_data
    GROUP BY 1
    ORDER BY 1
""")

monthly_trend.show()

+-------------------+-----------+-------------------+
|      month_revenue|total_trips|            revenue|
+-------------------+-----------+-------------------+
|2002-12-01 00:00:00|          1|                0.0|
|2003-01-01 00:00:00|          1|                0.0|
|2004-04-01 00:00:00|          1|               12.3|
|2008-12-01 00:00:00|         15| 230.51999999999998|
|2009-01-01 00:00:00|         74| 1195.0400000000002|
|2021-05-01 00:00:00|         29|             857.46|
|2021-06-01 00:00:00|    2834179|5.501827070023689E7|
|2021-07-01 00:00:00|    2821426|5.608368226019929E7|
|2021-08-01 00:00:00|         36|  898.0200000000002|
|2021-09-01 00:00:00|          3| 34.900000000000006|
|2021-10-01 00:00:00|          3| 53.599999999999994|
|2021-11-01 00:00:00|          5|              69.82|
|2021-12-01 00:00:00|          5|             120.41|
|2029-05-01 00:00:00|          1|              12.96|
+-------------------+-----------+-------------------+



In [14]:
# Melakukan Join dengan data taxi zone lookup

# 1. Load Data Zones
df_zones = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv('../data/raw/taxi_zone_lookup.csv')

# 2. Registrasi View untuk SQL
df_zones.createOrReplaceTempView('zones_data')

# 3. Query SQL untuk Clean + Join + Aggregate
# Filtering data untuk bulan Juni dan Juli saja
final_report = spark.sql("""
    SELECT
        z.Borough,
        z.Zone,
        tpep_pickup_datetime,
        COUNT(1) as total_trips,
        SUM(t.total_amount) as total_revennue
    FROM
        trips_data t
    JOIN
        zones_data z ON t.PULocationID = z.LocationID
    WHERE
        t.tpep_pickup_datetime >= '2021-06-01'
        AND t.tpep_pickup_datetime < '2021-08-01'
    GROUP BY
        1,2,3
    ORDER BY
        5 DESC
""")

final_report.show(10)

+---------+--------------------+--------------------+-----------+--------------+
|  Borough|                Zone|tpep_pickup_datetime|total_trips|total_revennue|
+---------+--------------------+--------------------+-----------+--------------+
|   Queens|   LaGuardia Airport| 2021-07-27 13:08:58|          1|        1320.8|
|Manhattan|            Kips Bay| 2021-07-31 11:18:09|          1|        988.35|
|      N/A|      Outside of NYC| 2021-06-26 21:31:19|          1|        958.05|
|Manhattan|Penn Station/Madi...| 2021-07-17 10:25:22|          1|        838.05|
|   Queens|   LaGuardia Airport| 2021-06-15 23:57:37|          1|        736.85|
|Manhattan|TriBeCa/Civic Center| 2021-06-04 21:20:18|          1|        722.97|
|   Queens|         JFK Airport| 2021-07-03 18:37:20|          1|        720.46|
|   Queens|            Flushing| 2021-07-15 19:04:29|          1|         700.3|
|   Queens|            Flushing| 2021-07-15 19:06:27|          1|         700.3|
|  Unknown|                 